# 01 - Extracao inicial de CNPJ - MG

Nesta etapa, iniciei a estrutura da extracao dos dados publicos de CNPJ para Minas Gerais, mantendo um conjunto inicial de campos relevantes para prospeccao B2B.

O objetivo deste primeiro notebook foi organizar os diretorios, definir o layout da base bruta e criar um CSV inicial para validar a estrutura antes de avancar para a carga dos arquivos completos da Receita Federal.

## 0. Objetivo da etapa

Nesta primeira etapa da coleta, preparei a base de trabalho para receber os dados de CNPJ. A prioridade foi deixar a estrutura simples, rastreavel e facil de validar antes de processar arquivos maiores.

Principais entregas desta etapa:

- Organizacao das pastas de entrada e saida dos dados de CNPJ.
- Definicao dos campos iniciais da base bruta.
- Criacao de um CSV vazio com os cabecalhos definidos.
- Preparacao do esqueleto para a proxima etapa de leitura dos arquivos publicos.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cnpj"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "cnpj"

OUTPUT_CSV = RAW_DIR / "cnpj_mg_extracao.csv"

RAW_DIR, PROCESSED_DIR, OUTPUT_CSV

## 1. Definição dos campos da base bruta

Defini um conjunto inicial de colunas combinando dados cadastrais, localizacao, contato, atividade economica e metadados da extracao. Esses campos sao suficientes para iniciar o garimpo e deixam espaco para enriquecimentos posteriores na etapa de tratamento.

In [ ]:
CNPJ_COLUMNS = [
    "cnpj_basico",
    "cnpj_ordem",
    "cnpj_dv",
    "cnpj",
    "nome_empresarial",
    "nome_fantasia",
    "matriz_filial",
    "situacao_cadastral",
    "data_situacao_cadastral",
    "motivo_situacao_cadastral",
    "data_inicio_atividade",
    "cnae_fiscal_principal",
    "cnae_fiscal_descricao",
    "cnae_fiscal_secundaria",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "uf",
    "municipio_codigo",
    "municipio_nome",
    "ddd_1",
    "telefone_1",
    "ddd_2",
    "telefone_2",
    "email",
    "capital_social",
    "porte_empresa",
    "natureza_juridica",
    "opcao_simples",
    "opcao_mei",
    "fonte_arquivo",
    "data_extracao",
]

len(CNPJ_COLUMNS)

## 2. Organização dos diretorios

Separei os arquivos de CNPJ em subpastas especificas dentro de `data/raw` e `data/processed`. Essa organizacao evita misturar a base da Receita Federal com futuras bases auxiliares, como dados do IBGE ou tabelas de apoio.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pasta raw: {RAW_DIR}")
print(f"Pasta processed: {PROCESSED_DIR}")

## 3. Criacao do CSV inicial

Com os campos definidos, criei um CSV inicial contendo apenas os cabecalhos. Esse arquivo funciona como contrato da base bruta e facilita a validacao do formato antes da extracao dos dados completos.

In [ ]:
if OUTPUT_CSV.exists():
    print(f"CSV ja existe: {OUTPUT_CSV}")
else:
    pd.DataFrame(columns=CNPJ_COLUMNS).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"CSV criado: {OUTPUT_CSV}")

## 4. Preparacao para a leitura dos arquivos publicos

A proxima etapa sera conectar esta estrutura aos arquivos publicos baixados da Receita Federal. Mantive abaixo os caminhos previstos para estabelecimentos, empresas e tabelas de apoio, que serao usados no cruzamento e nos filtros iniciais.

In [ ]:
ESTABELECIMENTOS_PATH = RAW_DIR / "estabelecimentos.csv"
EMPRESAS_PATH = RAW_DIR / "empresas.csv"
MUNICIPIOS_PATH = RAW_DIR / "municipios.csv"
CNAES_PATH = RAW_DIR / "cnaes.csv"

# Exemplo de filtro inicial previsto:
# - UF == "MG"
# - situacao_cadastral == "02"  # ativa na base da Receita
# - campos principais para prospeccao

future_inputs = {
    "estabelecimentos": ESTABELECIMENTOS_PATH,
    "empresas": EMPRESAS_PATH,
    "municipios": MUNICIPIOS_PATH,
    "cnaes": CNAES_PATH,
}

future_inputs

## 5. Validacao da estrutura criada

Por fim, conferi se o CSV criado possui exatamente as colunas definidas no notebook. Essa validacao reduz o risco de seguir para a etapa de extracao com um layout inconsistente.

In [ ]:
df_header = pd.read_csv(OUTPUT_CSV, nrows=0)

missing_columns = set(CNPJ_COLUMNS) - set(df_header.columns)
extra_columns = set(df_header.columns) - set(CNPJ_COLUMNS)

print(f"Total de colunas no CSV: {len(df_header.columns)}")
print(f"Colunas faltantes: {sorted(missing_columns)}")
print(f"Colunas extras: {sorted(extra_columns)}")

df_header.head()